# Step 4b · What exactly did the network features help with?

In Step 4 the "network" model found collapsing county-weeks better than the "local" model (P2 held), but the network feature that contributed most is one that **does not change with the storm**: the usual share of visitors from other counties. It mattered in Isaias too, which had no county-week with 72+ hours of outage, so it probably predicts swings unrelated to the storm (a tourist county losing visitors after summer, say). This step takes the network features apart:

| Model | Features |
| --- | --- |
| Local | As in Step 4: outage, wind and rain, county attributes |
| + dynamic exposure | Local + the four network exposures that change with outages (the outage this week in the counties visitors come from and residents usually visit) |
| + static shares | Local + the two network positions that do not change with the storm (usual share of visitors from other counties, usual share of residents' visits made elsewhere) |
| All | Step 4's "network" model |

The evaluation set (weeks 0–2) is then split in two:

- **Affected**: the county had 24+ hours of outage that week, or 0.5%+ of its usual visitors come from counties with 72+ hours, or 0.5%+ of the other counties residents usually visit had 72+ hours. Only here can a "large drop" be related to outages
- **Unaffected**: all other county-weeks. A "large drop" here has nothing to do with outages

Finally, Step 4's wording of P4 is corrected and the most underestimated county-weeks are listed.

**Structure**: A data (the same six cells as Step 4) → B the decision rules, fixed in advance → C four models → D average precision with county resampling → E the most underestimated county-weeks and P4 → F checklist and conclusions. Runs in a few minutes.

## A. Data and features (exactly as in Step 4)

The next six cells are copied unchanged from Step 4, so features, sample and parameters are the same.

In [ ]:
# 1. Environment, folders, chart style
import os, sys, glob, zipfile, subprocess
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance, partial_dependence
from sklearn.metrics import average_precision_score
try:
    import geopandas as gpd
except ImportError:                     # Colab usually has geopandas; install it if not
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas"], check=True)
    import geopandas as gpd

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive"
except ImportError:
    BASE = "."
OUT = f"{BASE}/outage_mobility"
for f in ["step1b_events.csv", "step1b_outage_event.csv", "step1b_panel_weekly.csv.gz", "step2a_hazard_weekly.csv",
          "geods_weeks/week_totals.csv", "oi_geoids_county.csv"]:
    if not os.path.exists(f"{OUT}/{f}"):
        raise FileNotFoundError(f"{OUT}/{f} is missing; run the earlier steps first.")

def tidy(df, n=2):
    # round numeric columns only, to keep tables readable
    return df.round({c: n for c in df.select_dtypes("number").columns})

INK, MUTED, GRID, SURFACE, FAINT = "#0b0b0b", "#52514e", "#e6e5e1", "#fcfcfb", "#b9b8b2"
C_DARK, C_MID, C_LIGHT, C_GREY = "#104281", "#2a78d6", "#86b6ef", "#7a7974"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "font.size": 10, "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
})

In [ ]:
# 2. Parameters
TRANSFER = "uri2021"                           # used only to test transfer, never for training
WEEK_FROM, WEEK_TO = -5, 6
BASE_COLS = [-5, -4, -3, -2]
TRAIN_FROM = -1                                # training uses weeks −1 to 6 (baseline weeks are near 0 by construction and are left out)
EVAL_WEEKS = [0, 1, 2]                         # evaluation looks only at the first three weeks after the storm
Y_CLIP = 150                                   # the outcome is clipped at ±150 log points to limit extreme noise
BIG_DROP = -20                                 # a "large drop": 20+ log points (about 18%) below normal counties in the same week
RURAL_POP = 50_000                             # counties below this population count as rural
AFFECTED_HOURS = 24                            # P4 looks only at county-weeks with 24+ hours of outage
HIT_HOURS = 72
B_BOOT = 999
SEED = 2026
GBM = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=50, l2_regularization=1.0, random_state=SEED)
# hyperparameters are fixed in advance with no tuning: tuning would use the held-out storm, which would then no longer be unseen
OUT_EDGES  = [0, 6, 24, 72, 120, np.inf]
OUT_LEVELS = ["h000", "h006", "h024", "h072", "h120"]
CUM_FROM = -1

In [ ]:
# 3. Sample, outcome, outage and hazard features
EVENTS = pd.read_csv(f"{OUT}/step1b_events.csv", index_col=0)
EV_ALL = list(EVENTS.index)
EV_TRAIN = [k for k in EV_ALL if k != TRANSFER]
W  = pd.read_csv(f"{OUT}/step1b_panel_weekly.csv.gz", dtype={"fips": str})
L  = pd.read_csv(f"{OUT}/step1b_outage_event.csv", dtype={"fips": str})
HZ = pd.read_csv(f"{OUT}/step2a_hazard_weekly.csv", dtype={"fips": str})

HZ = HZ.sort_values(["event", "fips", "rel_week"])
after = HZ["rel_week"] >= CUM_FROM
grp = [HZ["event"], HZ["fips"]]
HZ["gust_cum"] = HZ["gust_mean_ms"].where(after).groupby(grp).cummax().fillna(HZ["gust_mean_ms"])
HZ["rain_cum"] = HZ["rain_mm"].where(after, 0).groupby(grp).cumsum().where(after, HZ["rain_mm"])
D = (W.merge(L[["event", "fips", "name", "group", "flag_base", "flag_prior"]], on=["event", "fips"])
      .merge(HZ[["event", "fips", "rel_week", "gust_mean_ms", "rain_mm", "cold_ppt_mm", "tmin_c", "gust_cum", "rain_cum"]],
             on=["event", "fips", "rel_week"], how="left")
      .sort_values(["event", "fips", "rel_week"]).reset_index(drop=True))
g = D.groupby(["event", "fips"])["out_week_hours"]
D["out_prev_hours"] = g.shift(1)
D["out_cum_hours"] = D["out_week_hours"].where(D["rel_week"] >= CUM_FROM, 0).groupby([D["event"], D["fips"]]).cumsum()
D["out_bin"] = pd.cut(D["out_week_hours"], OUT_EDGES, right=False, labels=OUT_LEVELS).astype(str)
D["keep"] = ~D["flag_base"] & ~D["flag_prior"] & ~D["low_sample"] & D["out_week_hours"].notna() & D["visits_in_chg"].notna()
REF = D[D["keep"] & (D["group"] == "none")].groupby(["event", "rel_week"])["visits_in_chg"].median().rename("ref")
D = D.join(REF, on=["event", "rel_week"])
D["y"] = (D["visits_in_chg"] - D["ref"]).clip(-Y_CLIP, Y_CLIP)       # how much more it fell than normal counties that week
print(f"Reference counties (no substantial outage over the event) per event-week, minimum: "
      f"{int(D[D['keep'] & (D['group'] == 'none')].groupby(['event', 'rel_week']).size().min())}")

In [ ]:
# 4. County attributes: population, area, distance to the coast
GEO = f"{OUT}/step2c_geo"
COUNTY_URL = "https://www2.census.gov/geo/tiger/GENZ2020/shp/cb_2020_us_county_500k.zip"

def county_table():
    # use the county boundaries downloaded in Step 2c (only the land area in the attribute table); download them if missing
    path = f"{GEO}/county.zip"
    if not os.path.exists(path) or os.path.getsize(path) < 10_000:
        os.makedirs(GEO, exist_ok=True)
        r = requests.get(COUNTY_URL, timeout=600, headers={"User-Agent": "Mozilla/5.0 (research script)"})
        r.raise_for_status()
        with open(path, "wb") as fh:
            fh.write(r.content)
    folder = f"{GEO}/county"
    with zipfile.ZipFile(path) as z:
        z.extractall(folder)
    t = gpd.read_file(glob.glob(f"{folder}/*.shp")[0], ignore_geometry=True)
    return pd.Series(t["ALAND"].astype(float).to_numpy() / 1e6, index=t["GEOID"].astype(str), name="area_km2")

AREA = county_table()
OI = pd.read_csv(f"{OUT}/oi_geoids_county.csv")
OI["fips"] = OI["countyfips"].astype(int).astype(str).str.zfill(5)
POP = OI.set_index("fips")["county_pop2019"].astype(float)
COAST_FILE = f"{OUT}/step2c_coast_distance.csv"
COAST = pd.read_csv(COAST_FILE, dtype={"fips": str}).set_index("fips")["coast_km"] if os.path.exists(COAST_FILE) else pd.Series(dtype=float)
D["pop"] = D["fips"].map(POP)
D["log_pop"] = np.log(D["pop"])
D["log_density"] = np.log(D["pop"] / D["fips"].map(AREA))
D["coast_km"] = D["fips"].map(COAST)
D["rural"] = D["pop"] < RURAL_POP
print(f"County-weeks with population: {D['pop'].notna().mean():.1%}; with area: {D['fips'].map(AREA).notna().mean():.1%}; with distance to the coast: {D['coast_km'].notna().mean():.1%}")

In [ ]:
# 5. Network features (county-pair flows of each event)
TOT = pd.read_csv(f"{OUT}/geods_weeks/week_totals.csv", parse_dates=["week"]).set_index("week")["total_visits"].astype(float)
hours_all = W[["event", "fips", "rel_week", "out_week_hours"]].fillna({"out_week_hours": 0.0})
NET, STATIC = [], []
for key in EV_ALL:
    area = set(D.loc[D["event"] == key, "fips"])
    E = pd.read_csv(f"{OUT}/step1b_edges_{key}.csv.gz", dtype={"geoid_o": str, "geoid_d": str}, parse_dates=["week"])
    E = E[E["rel_week"].isin(BASE_COLS)]
    E["f"] = E["visitor_flows"] / E["week"].map(TOT).to_numpy() * 1e6
    B = (E.groupby(["geoid_o", "geoid_d"])["f"].sum() / len(BASE_COLS)).rename("f").reset_index()   # mean weekly flow over the baseline
    home = B["geoid_o"] == B["geoid_d"]
    res_tot = B[B["geoid_o"].isin(area)].groupby("geoid_o")["f"].sum()
    out_tot = B[~home & B["geoid_o"].isin(area)].groupby("geoid_o")["f"].sum()
    in_tot = B[B["geoid_d"].isin(area)].groupby("geoid_d")["f"].sum()
    in_non = B[~home & B["geoid_d"].isin(area)].groupby("geoid_d")["f"].sum()
    STATIC.append(pd.DataFrame({"event": key, "fips": sorted(area)}).assign(
        base_out_share=lambda d: d["fips"].map(out_tot / res_tot),
        base_visitor_share=lambda d: d["fips"].map(in_non / in_tot),
        log_base_visits_pc=lambda d: np.log(d["fips"].map(in_tot) / d["fips"].map(POP))))
    hrs = hours_all[hours_all["event"] == key].drop(columns="event")
    po = B[~home & B["geoid_o"].isin(area)].copy()
    po["w"] = po["f"] / po["geoid_o"].map(out_tot)                               # residents' usual split across other counties
    po = po.merge(hrs.rename(columns={"fips": "geoid_d"}), on="geoid_d")          # destinations outside the study area have no outage data and drop out of the sums (= treated as having power)
    po["w72"] = po["w"] * (po["out_week_hours"] >= HIT_HOURS)
    po["wh"] = po["w"] * po["out_week_hours"]
    outb = po.groupby(["geoid_o", "rel_week"])[["wh", "w72"]].sum().rename(columns={"wh": "net_out_hours", "w72": "net_out_72"})
    pi = B[~home & B["geoid_d"].isin(area)].copy()
    pi["w"] = pi["f"] / pi["geoid_d"].map(in_tot)                                 # usual share of visitors from each county
    pi = pi.merge(hrs.rename(columns={"fips": "geoid_o"}), on="geoid_o")
    pi["w72"] = pi["w"] * (pi["out_week_hours"] >= HIT_HOURS)
    pi["wh"] = pi["w"] * pi["out_week_hours"]
    inb = pi.groupby(["geoid_d", "rel_week"])[["wh", "w72"]].sum().rename(columns={"wh": "net_in_hours", "w72": "net_in_72"})
    both = outb.rename_axis(["fips", "rel_week"]).join(inb.rename_axis(["fips", "rel_week"]), how="outer").reset_index().assign(event=key)
    NET.append(both)
    print(f"{key}: {len(B):,} baseline county pairs")
NET = pd.concat(NET, ignore_index=True)
STATIC = pd.concat(STATIC, ignore_index=True)
D = D.drop(columns=[c for c in NET.columns if c.startswith("net_")] + ["base_out_share", "base_visitor_share", "log_base_visits_pc"], errors="ignore")
D = D.merge(NET, on=["event", "fips", "rel_week"], how="left").merge(STATIC, on=["event", "fips"], how="left")
for c in ["net_out_hours", "net_out_72", "net_in_hours", "net_in_72"]:
    D[c] = D[c].fillna(0.0)

In [ ]:
# 6. Feature table
F_LOCAL = ["out_week_hours", "out_prev_hours", "out_cum_hours", "gust_mean_ms", "gust_cum", "rain_mm", "rain_cum", "cold_ppt_mm", "tmin_c",
           "rel_week", "log_pop", "log_density", "coast_km", "log_base_visits_pc"]
F_NET = F_LOCAL + ["base_visitor_share", "base_out_share", "net_in_hours", "net_in_72", "net_out_hours", "net_out_72"]
S = D[D["keep"] & D["rel_week"].between(TRAIN_FROM, WEEK_TO) & D["y"].notna()].reset_index(drop=True)
S["is_eval"] = S["rel_week"].isin(EVAL_WEEKS)
S["big"] = S["y"] <= BIG_DROP
summary = S[S["is_eval"]].groupby("event").agg(county_weeks=("y", "size"), big_drops=("big", "sum"), hit_72=("out_week_hours", lambda h: int((h >= HIT_HOURS).sum())))
print("Features with the most missing values (gradient-boosted trees handle missing values directly):",
      S[F_NET].isna().mean().sort_values(ascending=False).head(4).round(3).to_dict())
summary

In [ ]:
# 7. Step 4b parameters and models
EXPO_MIN = 0.005                               # "affected": network exposure of at least 0.5%
F_DYN  = F_LOCAL + ["net_in_hours", "net_in_72", "net_out_hours", "net_out_72"]
F_STAT = F_LOCAL + ["base_visitor_share", "base_out_share"]
SETS = {"p_local": F_LOCAL, "p_dyn": F_DYN, "p_stat": F_STAT, "p_net": F_NET}
LABELS = {"p_local": "Local", "p_dyn": "+ dynamic exposure", "p_stat": "+ static shares", "p_net": "All (Step 4 network model)"}

## B. The decision rules, fixed in advance

Written down before looking at any result. All 95% intervals for average precision (AP) come from 999 resamples of counties.

- **R1 Dynamic exposure helps**: on "affected" county-weeks, "+ dynamic exposure" has a higher AP than "local", with an interval for the difference that excludes 0, and higher in a majority of the storms with at least 3 large drops in the affected part
- **R2 The static shares predict swings unrelated to outages**: on "unaffected" county-weeks, "+ static shares" has a higher AP than "local", with an interval for the difference that excludes 0
- **R3 The model finds outage-driven collapses**: on "affected" county-weeks, the "all" model has R² > 0 and an AP at least 3 times the chance level

**How it goes into the write-up**:

- R1 holds: "outage exposure through the pre-storm mobility network improves the prediction of outage-driven collapses in activity"
- R1 fails and R2 holds: Step 4's network gain came mostly from swings unrelated to outages, and the write-up cannot credit it to network exposure
- R3 is the floor for the model itself; if it fails, the only claim is "the model mainly predicts ordinary swings"

In [ ]:
# 8. Four models, leave one storm out; check against Step 4's predictions
parts = []
for key in EV_TRAIN:
    tr, te = S[S["event"].isin(EV_TRAIN) & (S["event"] != key)], S[S["event"] == key].copy()
    for col, feats in SETS.items():
        te[col] = HistGradientBoostingRegressor(**GBM).fit(tr[feats], tr["y"]).predict(te[feats])
    parts.append(te)
LOSO = pd.concat(parts, ignore_index=True)
EVS = LOSO[LOSO["is_eval"]].copy()
EVS["affected"] = (EVS["out_week_hours"] >= AFFECTED_HOURS) | (EVS["net_in_72"] >= EXPO_MIN) | (EVS["net_out_72"] >= EXPO_MIN)
OLD = f"{OUT}/step4_predictions_loso.csv"
REPRO = None
if os.path.exists(OLD):
    old = pd.read_csv(OLD, dtype={"fips": str}).merge(LOSO[["event", "fips", "rel_week", "p_net"]], on=["event", "fips", "rel_week"], suffixes=("_s4", ""))
    REPRO = float((old["p_net_s4"] - old["p_net"]).abs().max())
    print(f"Largest difference from Step 4's \"network\" predictions: {REPRO:.2e} ({len(old):,} rows)")
EVS.groupby("affected").agg(county_weeks=("y", "size"), large_drops=("big", "sum"), storms=("event", "nunique"))

## D. Average precision and county resampling

For three subsets (all, affected, unaffected) × four models: AP, its 95% interval and the chance level; plus three contrasts (+ dynamic exposure, + static shares and all, each minus local) with their intervals. Each resample applies the same county weights to every subset, so the intervals of the differences account for the correlation between models.

In [ ]:
# 9. AP, R² and differences, resampling counties
SUBSETS = {"all": np.ones(len(EVS), bool), "affected": EVS["affected"].to_numpy(), "unaffected": ~EVS["affected"].to_numpy()}
big = EVS["big"].to_numpy()
y = EVS["y"].to_numpy()
P = {col: EVS[col].to_numpy() for col in SETS}

def ap(mask, col, w=None):
    b = big[mask]
    return average_precision_score(b, -P[col][mask], sample_weight=None if w is None else w[mask]) if b.any() else np.nan

def r2(mask, col):
    yy, pp = y[mask], P[col][mask]
    return 1 - ((yy - pp) ** 2).sum() / ((yy - yy.mean()) ** 2).sum()

cid = pd.factorize(EVS["fips"])[0]
rng = np.random.default_rng(SEED)
draws = {(s, col): [] for s in SUBSETS for col in SETS}
for _ in range(B_BOOT):
    w = np.bincount(rng.integers(0, cid.max() + 1, cid.max() + 1), minlength=cid.max() + 1)[cid].astype(float)
    for s, m in SUBSETS.items():
        for col in SETS:
            draws[(s, col)].append(ap(m, col, w) if big[m & (w > 0)].any() else np.nan)
rows = []
for s, m in SUBSETS.items():
    for col in SETS:
        d = np.array(draws[(s, col)])
        rows.append({"subset": s, "model": col, "ap": ap(m, col), "lo": np.nanpercentile(d, 2.5), "hi": np.nanpercentile(d, 97.5),
                     "chance": big[m].mean(), "r2": r2(m, col), "n": int(m.sum()), "n_big": int(big[m].sum())})
AP_TAB = pd.DataFrame(rows)
rows = []
for s in SUBSETS:
    for col in ["p_dyn", "p_stat", "p_net"]:
        d = np.array(draws[(s, col)]) - np.array(draws[(s, "p_local")])
        pt = AP_TAB.set_index(["subset", "model"]).loc[(s, col), "ap"] - AP_TAB.set_index(["subset", "model"]).loc[(s, "p_local"), "ap"]
        rows.append({"subset": s, "contrast": f"{col} - p_local", "diff": pt, "lo": np.nanpercentile(d, 2.5), "hi": np.nanpercentile(d, 97.5)})
DIFF_TAB = pd.DataFrame(rows)
AP_TAB.to_csv(f"{OUT}/step4b_ap.csv", index=False)
DIFF_TAB.to_csv(f"{OUT}/step4b_differences.csv", index=False)
display(tidy(AP_TAB.pivot(index="model", columns="subset", values="ap").reindex(list(SETS)), 3))
tidy(DIFF_TAB.pivot(index="contrast", columns="subset", values="diff"), 3)

In [ ]:
# 10. Decision rules
D_ = DIFF_TAB.set_index(["subset", "contrast"])
A_ = AP_TAB.set_index(["subset", "model"])
aff = EVS[EVS["affected"]]
per = []
for key, g in aff.groupby("event"):
    if g["big"].sum() >= 3:
        per.append({"event": key, "n_big": int(g["big"].sum()),
                    **{col: average_precision_score(g["big"], -g[col]) for col in SETS}})
PER = pd.DataFrame(per)
PER.to_csv(f"{OUT}/step4b_affected_by_storm.csv", index=False)
WINS, ELIG = int((PER["p_dyn"] > PER["p_local"]).sum()) if len(PER) else 0, len(PER)
R1 = bool(D_.loc[("affected", "p_dyn - p_local"), "lo"] > 0 and WINS > ELIG / 2)
R2 = bool(D_.loc[("unaffected", "p_stat - p_local"), "lo"] > 0)
R3 = bool(A_.loc[("affected", "p_net"), "r2"] > 0 and A_.loc[("affected", "p_net"), "ap"] >= 3 * A_.loc[("affected", "p_net"), "chance"])
for name, val, txt in [("R1", R1, "dynamic exposure helps"), ("R2", R2, "static shares predict swings unrelated to outages"),
                       ("R3", R3, "the model finds outage-driven collapses")]:
    print(f"{name} {txt}: {val}")
print(f"R1: affected AP difference {D_.loc[('affected', 'p_dyn - p_local'), 'diff']:+.3f} [{D_.loc[('affected', 'p_dyn - p_local'), 'lo']:+.3f}, {D_.loc[('affected', 'p_dyn - p_local'), 'hi']:+.3f}], better in {WINS} of {ELIG} storms")
print(f"R2: unaffected AP difference {D_.loc[('unaffected', 'p_stat - p_local'), 'diff']:+.3f} [{D_.loc[('unaffected', 'p_stat - p_local'), 'lo']:+.3f}, {D_.loc[('unaffected', 'p_stat - p_local'), 'hi']:+.3f}]")
if R1:
    print("Wording: outage exposure through the pre-storm mobility network improves the prediction of outage-driven collapses in activity")
elif R2:
    print("Wording: Step 4's network gain came mostly from swings unrelated to outages and cannot be credited to network exposure")
else:
    print("Wording: the role of the network features is unclear")
tidy(PER.set_index("event"), 3)

In [ ]:
# 11. Figure: AP of the four models on the three subsets; affected part by storm
COL = {"p_local": C_LIGHT, "p_dyn": C_MID, "p_stat": C_GREY, "p_net": C_DARK}
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1.3, 1]})
ax = axes[0]
subs = list(SUBSETS)
for i, s in enumerate(subs):
    for j, col in enumerate(SETS):
        r = A_.loc[(s, col)]
        x = i + (j - 1.5) * 0.16
        ax.errorbar(x, r["ap"], yerr=[[r["ap"] - r["lo"]], [r["hi"] - r["ap"]]], fmt="o", color=COL[col], ms=6, elinewidth=1.5, capsize=0,
                    label=LABELS[col] if i == 0 else None)
    ch = A_.loc[(s, "p_local"), "chance"]
    ax.plot([i - 0.36, i + 0.36], [ch, ch], color=INK, lw=1.2, label="Chance level" if i == 0 else None)
ax.set_xticks(range(len(subs)))
ax.set_xticklabels([f"{s}\n{int(A_.loc[(s, 'p_local'), 'n']):,} county-weeks\n{int(A_.loc[(s, 'p_local'), 'n_big'])} large drops" for s in subs])
ax.set_ylabel("Average precision for large drops")
ax.set_title("Which features help, and where (95% intervals, resampling counties)")
handles, labs = ax.get_legend_handles_labels()
ax.grid(axis="x", visible=False)
ax = axes[1]
if len(PER):
    yy = np.arange(len(PER))[::-1]
    for col in ["p_local", "p_dyn", "p_net"]:
        off = {"p_local": 0.18, "p_dyn": 0.0, "p_net": -0.18}[col]
        ax.scatter(PER[col], yy + off, color=COL[col], s=36, zorder=3, label=LABELS[col])
    ax.set_yticks(yy)
    ax.set_yticklabels([f"{EVENTS.loc[k, 'name']} ({n} large drops)" for k, n in zip(PER["event"], PER["n_big"])])
    ax.set_xlabel("Average precision, affected county-weeks")
    ax.set_title("Affected county-weeks, by held-out storm")
    ax.grid(axis="y", visible=False)
fig.legend(handles, labs, loc="lower center", ncol=5, fontsize=9, bbox_to_anchor=(0.5, 0.0))
fig.tight_layout(rect=(0, 0.07, 1, 1))
fig.savefig(f"{OUT}/step4b_fig1_ablation.png"); plt.show()

## E. The most underestimated county-weeks, and the correct wording of P4

Step 4's P4 compared "mean residual of rural counties − mean residual of other counties". Residual = actual − predicted, so **the more negative the residual, the more the loss was underestimated**. Hence:

- difference > 0 (interval excluding 0): **the losses of other (larger) counties are underestimated more**
- difference < 0 (interval excluding 0): **the losses of rural counties are underestimated more**

Step 4 wrote "difference > 0" as "overestimates rural losses", which is arithmetically true but easy to misread; it is rewritten here as above, with how much each group's losses were underestimated.

In [ ]:
# 12. The most underestimated county-weeks; the correct wording of P4
names = D.drop_duplicates(["event", "fips"]).set_index(["event", "fips"])["name"]
worst = EVS[EVS["affected"]].assign(residual=lambda d: d["y"] - d["p_net"]).nsmallest(12, "residual")
worst = worst.assign(county=[names.get((e, f), f) for e, f in zip(worst["event"], worst["fips"])],
                     storm=worst["event"].map(EVENTS["name"]), rural=worst["rural"].map({True: "rural", False: "other"}))
worst[["storm", "county", "rel_week", "out_week_hours", "y", "p_net", "residual", "rural"]].to_csv(f"{OUT}/step4b_most_underestimated.csv", index=False)
display(tidy(worst[["storm", "county", "rel_week", "out_week_hours", "y", "p_net", "residual", "rural"]], 1))

EQ = pd.read_csv(f"{OUT}/step4_equity.csv") if os.path.exists(f"{OUT}/step4_equity.csv") else None
if EQ is not None:
    for label, g in EQ.groupby("sample", sort=False):
        gap_, lo_, hi_ = g.iloc[0][["gap_rural_minus_other", "gap_lo", "gap_hi"]]
        res = g.set_index("group")["mean_residual"]
        if lo_ > 0:
            txt = "the losses of other (larger) counties are underestimated more"
        elif hi_ < 0:
            txt = "the losses of rural counties are underestimated more"
        else:
            txt = "no systematic difference in how much the two groups are underestimated"
        print(f"{label}: rural mean residual {res.get('rural (< 50k)', np.nan):+.1f}, other counties {res.get('other', np.nan):+.1f}; "
              f"difference {gap_:+.1f} [{lo_:+.1f}, {hi_:+.1f}] → {txt}")

In [ ]:
# 13. Checklist and conclusions
def verdict(passed):
    return "pending" if passed is None else ("pass" if passed else "fail")

checks = [
    ("\"All\" model matches Step 4's predictions (largest difference)", "—" if REPRO is None else f"{REPRO:.1e}", "< 1e-6",
     None if REPRO is None else REPRO < 1e-6),
    ("Large drops in the affected part", int(A_.loc[("affected", "p_local"), "n_big"]), "≥ 30", bool(A_.loc[("affected", "p_local"), "n_big"] >= 30)),
    ("Large drops in the unaffected part", int(A_.loc[("unaffected", "p_local"), "n_big"]), "≥ 30", bool(A_.loc[("unaffected", "p_local"), "n_big"] >= 30)),
]
check_tab = pd.DataFrame(checks, columns=["Indicator", "Value", "Threshold", "passed"])
check_tab["Verdict"] = check_tab.pop("passed").map(verdict)
check_tab.to_csv(f"{OUT}/step4b_checklist.csv", index=False)
display(check_tab)

def yes_no(x):
    return "yes" if x else "no"
fmt = lambda s, c: f"{D_.loc[(s, c), 'diff']:+.3f} [{D_.loc[(s, c), 'lo']:+.3f}, {D_.loc[(s, c), 'hi']:+.3f}]"
a = A_.loc[("affected", "p_net")]
conclusions = pd.DataFrame([
    ("R1", f"affected AP difference {fmt('affected', 'p_dyn - p_local')}; {WINS}/{ELIG}", yes_no(R1)),
    ("R2", f"unaffected AP difference {fmt('unaffected', 'p_stat - p_local')}", yes_no(R2)),
    ("R3", f"R² {a['r2']:.3f}; AP {a['ap']:.3f} (chance {a['chance']:.3f})", yes_no(R3)),
], columns=["Question", "Value", "Conclusion"])
conclusions.to_csv(f"{OUT}/step4b_conclusions.csv", index=False)
conclusions